In [1]:
import pandas as pd
import requests
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
from urllib.parse import quote

In [2]:
#load our results
df = pd.read_excel("../data/drugs_for_each_case_in_one_sheet.xlsx")
drugs_list=df["Drug"].unique()

In [3]:
#functions to find smiles and synonyms for our results
def get_smiles(name):
    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{quote(name)}/property/CanonicalSMILES/TXT"
    )

    r = requests.get(url, timeout=10)

    if r.status_code == 200:
        return r.text.strip()

    return None


def get_synonyms(name):
    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{quote(name)}/synonyms/JSON"
    )

    r = requests.get(url, timeout=10)
    if r.status_code != 200:
        return []

    try:
        return r.json()["InformationList"]["Information"][0]["Synonym"]
    except Exception:
        return []


def get_smiles_with_synonyms(name):

    smiles = get_smiles(name)

    if smiles:
        return smiles, name

    synonyms = get_synonyms(name)

    for syn in synonyms:
        smiles = get_smiles(syn)

        if smiles:
            return smiles, syn

    return None, None

In [4]:
#generate the synonyms and smiles for our results
results = {}

for drug in drugs_list:
    smiles, matched_name = get_smiles_with_synonyms(drug)

    results[drug] = {
            "matched_name": matched_name,
            "smiles": smiles
        }

In [5]:
#function to extract descriptors

def calc_descriptors(smiles):
    if not smiles or smiles is None:
        return None

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
    }

In [6]:
#function to calculate a computationally score
def bbb_score(d):
    if d is None:
        return None

    score = 0

    if d["TPSA"] < 90: score += 2
    if d["TPSA"] < 70: score += 1

    if 1 <= d["LogP"] <= 3.5: score += 2
    elif d["LogP"] < 5: score += 1

    if d["MW"] < 450: score += 1
    if d["HBD"] <= 3: score += 1
    if d["HBA"] <= 7: score += 1

    return score

In [7]:
# score results
results_perm = {}

for drug, info in results.items():
    smiles = info.get("smiles")

    desc = calc_descriptors(smiles)
    score = bbb_score(desc)

    results_perm[drug] = {
        **info,
        "descriptors": desc,
        "BBB_score": score
    }

In [8]:
#export results
rows = []
for drug, info in results_perm.items():
    d = info["descriptors"]

    rows.append({
        "drug": drug,
        "smiles": info.get("smiles"),
        "BBB_score": info["BBB_score"],
        "MW": None if d is None else d["MW"],
        "LogP": None if d is None else d["LogP"],
        "TPSA": None if d is None else d["TPSA"],
        "HBD": None if d is None else d["HBD"],
        "HBA": None if d is None else d["HBA"]
    })

df = pd.DataFrame(rows)
df.to_csv("../data/bbb_results.csv", index=False)